In [2]:
%load_ext autoreload
%autoreload 2

import torch
import torch._dynamo
torch._dynamo.config.suppress_errors = True

import numpy as np

import os
import pickle
from tqdm import tqdm
from glob import glob

import esm

from msa_model.dataset import MSA

device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
torch.set_float32_matmul_precision('high')
print(f"Using device: {torch.cuda.get_device_name(device)}")

/usr/lib/python3/dist-packages/scipy/__init__.py:146: UserWarning: A NumPy version >=1.17.3 and <1.25.0 is required for this version of SciPy (detected version 1.26.0
  warnings.warn(f"A NumPy version >={np_minversion} and <{np_maxversion}"


Using device: NVIDIA GH200 480GB


In [3]:
# Load target information
target_data_file = "../../data/Figure2_heterooligomer_contact_prediction/target_data.pkl"
with open(target_data_file, "rb") as oFile:
    target_data_d = pickle.load(oFile)


In [4]:
# Load model
model_path = "/home/ubuntu/esm_models/esm_msa1b_t12_100M_UR50S.pt"
msa_transformer, msa_transformer_alphabet = esm.pretrained.load_model_and_alphabet_local(model_path)
msa_transformer = msa_transformer.eval().to(torch.float16).cuda()
msa_transformer_batch_converter = msa_transformer_alphabet.get_batch_converter()
total_params = sum(p.numel() for p in msa_transformer.parameters())
print(f"Total number of parameters: {total_params:,}")

/home/ubuntu/.local/lib/python3.10/site-packages/esm/pretrained.py:70: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model_data = torch.load(str(model_location), map_locatio

Total number of parameters: 115,616,434


In [ ]:
max_tokens = np.inf
nSeqs = 512
results_dir = "results/msa_transformer/"
if not os.path.exists(results_dir):
    os.makedirs(results_dir)
for target_id in target_data_d.keys():
    # Load alignment
    msa_file_path = target_data_d[target_id]['msa_file_path']
    with open(msa_file_path, "r") as oFile:
        query_seq = oFile.readlines()[1].strip()
    len_seq = len(query_seq)
    if len_seq >= 1024:
        continue
    if len_seq >= 512:
        nSeqs = 256
    else:
        len_seq = 512
    np.random.seed(42)
    msa_obj = MSA(
        msa_file_path = msa_file_path,
        max_seqs = nSeqs,
        max_length = len_seq,
        max_tokens = max_tokens,
        diverse_select_method = "hhfilter",
        hhfilter_kwargs = {"qid": 30},
    )
    msa_seq_l = [(seq_idx, "".join(seq)) for seq_idx, seq in enumerate(msa_obj.select_diverse_msa)]
    # Prep MSA Transformer inputs
    batch_labels, batch_strs, batch_tokens = msa_transformer_batch_converter([msa_seq_l])
    batch_tokens = batch_tokens.to(device)
    # Predict contacts
    with torch.no_grad():
        predicted_contacts_a = msa_transformer.predict_contacts(batch_tokens)[0].cpu().numpy()
    # Save contacts
    np.savetxt(f"{results_dir}/{target_id}.txt", predicted_contacts_a)